# Make Combined Estimation File

This similar to the standard Make Estimation file, but it replaces the TNC trips observed in the HH travel survey with the TNC trips observed in the Chicago TNP data.  This is similar to replacing transit trips in a HH survey with those from an onboard transit survey.  It is a choice-based sampling approach that provides for a much larger sample size for trips with small shares that we care about.  


In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.reset_option('display.float_format')


In [3]:
# start from the survey trips

survey_trips = pd.read_csv('out/mode_choice_estimation_file.csv')
survey_trips.head()

,hh_id,person_id,person_num,day_id,day_num,joint_trip_id,joint_trip_num,depart_date,depart_hour,depart_minute,depart_seconds,arrive_date,arrive_hour,arrive_minute,arrive_second,distance_meters,distance_miles,duration_minutes,dwell_mins,flag_speed,flag_distance,flag_duration,o_tract_2020,d_tract_2020,hh_member_1,hh_member_2,hh_member_3,hh_member_4,hh_member_5,hh_member_6,hh_member_7,hh_member_8,hh_member_9,hh_member_10,o_purpose,o_purpose_category,d_purpose,d_purpose_category,n_legs,leg_num,first_leg,last_leg,linked_trip_id,linked_trip_num,linked_trip_mode,outbound,joint_status,linked_trip_weight,tour_id,tour_num,linked_trip_mode_labeled,mode,mode2,o_district,d_district,o_community,d_community,hh_id_tour,person_id_tour,person_num_tour,day_id_tour,day_num_tour,distance_meters_tour,distance_miles_tour,duration_minutes_tour,o_tract_2020_tour,d_tract_2020_tour,num_travelers,num_hh_travelers,joint_status_tour,tour_num_tour,joint_tour_id,tour_start_date,tour_start_hour,tour_start_minute,tour_start_second,tour_end_date,tour_end_hour,tour_end_minute,tour_end_second,tour_category,tour_mode,tour_purpose,partial_status,stops_outbound,stops_inbound,tour_weight,day_of_week,time_period,origin_id,destination_id,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,walk_faster_than_transit,transit_or_walk_fare,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk,transit_avail,walk_avail,transit_or_walk_avail,income_detailed,income_broad,num_vehicles,num_people,num_workers,num_adults,num_kids,hh_weight,income_labeled,tract_2020,hh_share_inc_under_100k,hh_share_inc_over_100k,tract_2020_dropoff,hh_share_inc_under_100k_dropoff,hh_share_inc_over_100k_dropoff
0,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,11,20,0.0,2024-05-21,11,40,0.0,1304,0.810270,20,5.0,0,0,0,17031320101,17031081500,1,0,0,0,0,0,0,0,0,0,1,1,33,10,4,1,1,0,2400012401010101,1,15,1,1,1853.792592,24000124010101,1,Walk,walk,walk,Downtown,Downtown,32.0,8.0,24000124,2400012401,1,240001240101,1,3246,2.016976,175,1.703132e+10,1.703132e+10,1,1,1,1,NaN,2024-05-21,11,20,0.0,2024-05-21,14,15,0.0,2,15,10,0,0,2,1237.449397,Tuesday,midday,17031320101,17031081500,3.738333,6.186942,5,11.186942,6.707731,2.5,16.205401,22.0,16.205401,True,0.0,-5.018459,6.707731,1,1,1,12,5,2,2,1,2,0,794.392,"$150,000 or more",17031320101,0.341000,0.659000,17031081500,0.321678,0.678322
1,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,11,45,0.0,2024-05-21,12,13,0.0,544,0.338027,28,82.0,0,0,0,17031081500,17031081403,1,0,0,0,0,0,0,0,0,0,33,10,150,12,4,2,0,0,2400012401010102,2,15,0,1,1853.792592,24000124010101,1,Walk,walk,walk,Downtown,Downtown,8.0,8.0,24000124,2400012401,1,240001240101,1,3246,2.016976,175,1.703132e+10,1.703132e+10,1,1,1,1,NaN,2024-05-21,11,20,0.0,2024-05-21,14,15,0.0,2,15,10,0,0,2,1237.449397,Tuesday,midday,17031081500,17031081403,1.660000,2.747300,5,7.747300,4.798943,2.5,6.760535,7.0,6.760535,True,0.0,0.986765,4.798943,1,1,1,12,5,2,2,1,2,0,794.392,"$150,000 or more",17031081500,0.321678,0.678322,17031081403,0.460539,0.539461
2,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,13,35,0.0,2024-05-21,13,50,0.0,884,0.549293,15,9.0,0,0,0,17031081403,17031320101,1,0,0,0,0,0,0,0,0,0,150,12,33,10,4,3,0,0,2400012401010103,3,15,0,1,1853.792592,24000124010101,1,Walk,walk,walk,Downtown,Downtown,8.0,32.0,24000124,2400012401,1,240001240101,1,3246,2.016976,175,1.703132e+10,1.703132e+10,1,1,1,1,NaN,2024-05-21,11,20,0.0,2024-05-21,14,15,0.0,2,15,10,0,0,2,1237.449397,Tuesday,midday,17031081403,17031320101,3.421667,5.662858,5,10.662858,6.244886,2.5,10.985870,23.0,10.985870,True,0.0,-0.323012,6.244886,1,1,1,12,5,2,2,1,2,0,794.392,"$150,000 or more",17031081403,0.460539,0.539461,17031320101,0.341000,0.659000
3,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,13,59,0.0,2024-05-21,14,15,0.0,514,0.319386,16,NaN,0,0,0,17031320101,17031320101,1,0,0,0,0,0,0,0,0,0,33,10,1,1,4,4,0,1,2400012401010104,4,15,0,1,1853.792592,24000124010101,1,Walk,walk,

In [4]:
# and the TNP trips

tnc_trips = pd.read_csv('../data_processing/out/processed_tnp_trips.csv')
tnc_trips.head()

,Trip ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Percent Time Chicago,Percent Distance Chicago,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,Fare,Tip,Additional Charges,Trip Total,Shared Trip Authorized,Shared Trip Match,Trips Pooled,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location,time_period,Inferred Pickup Census Tract (time),Inferred Dropoff Census Tract (time),Pickup district,Dropoff district,congested_car_time_min,fftime,calc_fare,transit_time,walk_time,Pickup hh_share_inc_under_100k,Pickup hh_share_inc_over_100k,Dropoff hh_share_inc_under_100k,Dropoff hh_share_inc_over_100k
0,f5045620992c5bb5cac200d5db51fbc9d1e980dc,04/16/2025 11:45:00 PM,04/17/2025 12:00:00 AM,396,1.0,99%,98%,1.703108e+10,1.703132e+10,8,32,$7.5,$0,$3.41,$10.91,False,False,1,41.892073,-87.628874,POINT (-87.6288741572 41.8920726347),41.884987,-87.620993,POINT (-87.6209929134 41.8849871918),night,17031081600,17031320100,Downtown,NaN,NaN,NaN,7.075723,NaN,20.0,0.319000,0.681000,NaN,NaN
1,f5189089ccfbbde63cc147ad6c15a35b7d3acb89,04/16/2025 11:45:00 PM,04/16/2025 11:45:00 PM,737,2.9,100%,100%,1.703106e+10,1.703103e+10,6,77,$5,$3,$3.13,$11.13,False,False,1,41.950673,-87.666536,POINT (-87.6665362813 41.9506733576),41.987226,-87.664938,POINT (-87.6649377243 41.9872255578),night,17031060400,17031030500,Other,Other,8.266137,6.941667,11.255542,75.0,58.0,0.267000,0.733000,0.348303,0.651697
2,f53b28d0530f1e3657d0741e2cf98574a3683b75,04/16/2025 11:45:00 PM,04/17/2025 12:00:00 AM,310,1.5,100%,100%,NaN,NaN,41,39,$7.5,$0,$2.31,$9.81,False,False,1,41.794090,-87.592311,POINT (-87.592310855 41.794090253),41.808916,-87.596183,POINT (-87.5961833442 41.8089162826),night,17031411100,17031390500,Other,Other,5.166087,4.338333,6.908713,40.0,30.0,0.426573,0.573427,0.594595,0.405405
3,f55bc6fbaee328d0ee6108316cb12daf270099b9,04/16/2025 11:45:00 PM,04/17/2025 12:15:00 AM,1243,8.8,100%,100%,NaN,NaN,4,33,$12.5,$0,$3.33,$15.83,False,False,1,41.975171,-87.687516,POINT (-87.6875155152 41.9751709433),41.857184,-87.620335,POINT (-87.6203346241 41.8571838585),night,17031040100,17031330200,Other,Other,20.717935,17.398333,20.248924,999.0,176.0,0.551000,0.449000,0.304695,0.695305
4,f56a5b36e42c9067ddf4ac0a40a9fb568034eb30,04/16/2025 11:45:00 PM,04/17/2025 12:00:00 AM,314,1.5,100%,100%,NaN,NaN,31,28,$7.5,$0,$1.23,$8.73,False,False,1,41.850266,-87.667569,POINT (-87.667569312 41.8502663663),41.874005,-87.663518,POINT (-87.6635175498 41.874005383),night,17031843200,17031832900,Other,Other,5.243489,4.403333,6.937551,49.0,30.0,0.567000,0.433000,0.585000,0.415000


In [5]:
# keep only the relevant fields
survey_trips = survey_trips[[
    'hh_id',
    'person_id',
    'person_num',
    'day_id',
    'day_num',
    'depart_date',
    'o_tract_2020',
    'd_tract_2020',
    'linked_trip_id',
    'linked_trip_num',
    'linked_trip_mode',
    'linked_trip_weight',
    'linked_trip_mode_labeled',
    'mode',
    'mode2',
    'distance_miles', 
    'duration_minutes', 
    'o_district',
    'd_district',
    'o_community',
    'd_community',
    'time_period',
    'ff_car_time_minutes',
    'car_ivt',
    'tnc_wait',
    'tnc_time',
    'tnc_fare',
    'transit_fare',
    'walk_time',
    'transit_time',
    'transit_or_walk_time',
    'walk_faster_than_transit',
    'transit_or_walk_fare',
    'income_broad',
    'income_labeled',
    'hh_share_inc_under_100k',
    'hh_share_inc_over_100k',
    'hh_share_inc_under_100k_dropoff',
    'hh_share_inc_over_100k_dropoff'
    ]]


In [6]:
# keep only the relevant fields in the TNP file
tnc_trips = tnc_trips[[
    'Trip ID',
    'Trip Seconds',
    'Trip Miles',
    'Fare',
    'Tip',
    'Additional Charges',
    'time_period',
    'Pickup district', 
    'Dropoff district', 
    'Pickup Community Area', 
    'Dropoff Community Area', 
    'Inferred Pickup Census Tract (time)',
    'Inferred Dropoff Census Tract (time)',
    'fftime', 
    'congested_car_time_min',
    'calc_fare',
    'transit_time',
    'walk_time',
    'Pickup hh_share_inc_under_100k',
    'Pickup hh_share_inc_over_100k',
    'Dropoff hh_share_inc_under_100k',
    'Dropoff hh_share_inc_over_100k'
    ]]



In [7]:
# align the columns such that key fields have the same names

tnc_trips = tnc_trips.rename(columns = {'Trip ID' : 'tnc_trip_id', 
                                        'Fare' : 'obs_fare', 
                                        'Tip' : 'obs_tip', 
                                        'Additional Charges' : 'obs_additional_charges', 
                                        'Trip Miles' : 'distance_miles', 
                                        'Trip Seconds' : 'duration_seconds',                  
                                        'Pickup district' : 'o_district', 
                                        'Dropoff district' : 'd_district',            
                                        'Pickup Community Area' : 'o_community', 
                                        'Dropoff Community Area' : 'd_community', 
                                        'Inferred Pickup Census Tract (time)' : 'o_tract_2020',
                                        'Inferred Dropoff Census Tract (time)' : 'd_tract_2020',
                                        'fftime' : 'ff_car_time_minutes',
                                        'congested_car_time_min' : 'car_ivt',
                                        'calc_fare' : 'tnc_fare',
                                        'Pickup hh_share_inc_under_100k': 'hh_share_inc_under_100k',
                                        'Pickup hh_share_inc_over_100k' : 'hh_share_inc_over_100k',
                                        'Dropoff hh_share_inc_under_100k': 'hh_share_inc_under_100k_dropoff',
                                        'Dropoff hh_share_inc_over_100k' : 'hh_share_inc_over_100k_dropoff'
                                        })

In [8]:
# calculate additional columns for better alignment

tnc_trips['duration_minutes'] = tnc_trips['duration_seconds'] / 60
tnc_trips = tnc_trips.drop(columns=['duration_seconds'])

tnc_trips['mode'] = 'tnc'
tnc_trips['mode2'] = 'tnc'
tnc_trips['tnc_wait'] = 5
tnc_trips['tnc_time'] = tnc_trips['tnc_wait'] + tnc_trips['car_ivt']
tnc_trips['transit_fare'] = 2.5
tnc_trips['linked_trip_weight'] = 1.0



In [9]:
# convert observed fares to numerical columns
for col in ['obs_fare', 'obs_tip', 'obs_additional_charges']:
    tnc_trips[col] = (
        tnc_trips[col]
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )


In [10]:
# drop the TNC trips from the HH survey and combine the files
survey_trips = survey_trips[survey_trips['mode'] != 'tnc'].copy()
combined_trips = pd.concat([survey_trips, tnc_trips], ignore_index=True)
combined_trips

,hh_id,person_id,person_num,day_id,day_num,depart_date,o_tract_2020,d_tract_2020,linked_trip_id,linked_trip_num,linked_trip_mode,linked_trip_weight,linked_trip_mode_labeled,mode,mode2,distance_miles,duration_minutes,o_district,d_district,o_community,d_community,time_period,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,walk_faster_than_transit,transit_or_walk_fare,income_broad,income_labeled,hh_share_inc_under_100k,hh_share_inc_over_100k,hh_share_inc_under_100k_dropoff,hh_share_inc_over_100k_dropoff,tnc_trip_id,obs_fare,obs_tip,obs_additional_charges
0,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031081500,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.810270,20.000000,Downtown,Downtown,32.0,8.0,midday,3.738333,6.186942,5,11.186942,6.707731,2.5,16.205401,22.0,16.205401,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0.321678,0.678322,NaN,NaN,NaN,NaN
1,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081500,17031081403,2.400012e+15,2.0,15.0,1853.792592,Walk,walk,walk,0.338027,28.000000,Downtown,Downtown,8.0,8.0,midday,1.660000,2.747300,5,7.747300,4.798943,2.5,6.760535,7.0,6.760535,True,0.0,5.0,"$150,000 or more",0.321678,0.678322,0.460539,0.539461,NaN,NaN,NaN,NaN
2,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081403,17031320101,2.400012e+15,3.0,15.0,1853.792592,Walk,walk,walk,0.549293,15.000000,Downtown,Downtown,8.0,32.0,midday,3.421667,5.662858,5,10.662858,6.244886,2.5,10.985870,23.0,10.985870,True,0.0,5.0,"$150,000 or more",0.460539,0.539461,0.341000,0.659000,NaN,NaN,NaN,NaN
3,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320101,2.400012e+15,4.0,15.0,1853.792592,Walk,walk,walk,0.319386,16.000000,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.167457,2.5,6.387712,12.0,6.387712,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0.341000,0.659000,NaN,NaN,NaN,NaN
4,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320102,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.751861,16.000000,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.561010,2.5,15.037220,12.0,12.000000,False,2.5,5.0,"$150,000 or more",0.341000,0.659000,0.519520,0.480480,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188180,NaN,NaN,NaN,NaN,NaN,NaN,17031839000,17031071600,NaN,NaN,NaN,1.000000,NaN,tnc,tnc,3.400000,10.116667,Downtown,Other,32.0,7.0,night,8.488333,10.107907,5,15.107907,10.771317,2.5,68.000000,95.0,NaN,NaN,NaN,NaN,NaN,0.485000,0.515000,0.266266,0.733734,19d5bdd87861afc50be6e60d86bb6162eab1d37f,27.5,0.0,3.55
188181,NaN,NaN,NaN,NaN,NaN,NaN,17031831500,17031190100,NaN,NaN,NaN,1.000000,NaN,tnc,tnc,2.300000,10.783333,Other,Other,19.0,19.0,night,8.048333,9.583955,5,14.583955,10.063092,2.5,46.000000,74.0,NaN,NaN,NaN,NaN,NaN,0.567000,0.433000,0.775551,0.224449,19e320f52f745a31fb9e936bb81054ff717a18f9,5.0,0.0,1.45
188182,NaN,NaN,NaN,NaN,NaN,NaN,17031330101,17031430200,NaN,NaN,NaN,1.000000,NaN,tnc,tnc,8.500000,15.766667,Tourist,Other,33.0,43.0,night,13.246667,15.774131,5,20.774131,17.835923,2.5,170.000000,999.0,NaN,NaN,NaN,NaN,NaN,0.197605,0.802395,0.816000,0.184000,1a450fa5be8973044470459fb0996f917539193c,15.0,0.0,3.63
188183,NaN,NaN,NaN,NaN,NaN,NaN,17031460400,17031710600,NaN,NaN,NaN,1.000000,NaN,tnc,tnc,5.400000,19.316667,Other,Other,46.0,71.0,night,11.526667,13.725955,5,18.725955,16.562925,2.5,108.000000,147.0,NaN,NaN,NaN,NaN,NaN,0.715431,0.284569,0.875250,0.124750,1acd6586d7fa041e6f6ba72c92ce334848422db5,15.0,0.0,1.23


# Filter records

Exclude records where there are illogical results. 

In [11]:
# Keep only trips in Chicago

trips_before = len(combined_trips)
combined_trips = combined_trips[(combined_trips['o_district'] != 'External') & (combined_trips['d_district'] != 'External') ].copy()
print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 188185 After: 188139


In [12]:
# drop trips where the calculated transit time is missing.  These seem to be trips that go out of state
# and where we didn't calculate paths. 

trips_before = len(combined_trips)
combined_trips = combined_trips[combined_trips['transit_time'].notna()].copy()
print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 188139 After: 174805


In [13]:
# next we filter to exclude trips we don't have in our TNC data.
# We could re-consider whether to drop the airport trips.  It's not obvious it's necessary to drop them, but there are not many here. 

trips_before = len(combined_trips)
combined_trips = combined_trips[(combined_trips['o_district'] != 'Airport') & (combined_trips['d_district'] != 'Airport') ].copy()
print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 174805 After: 161874


In [14]:
# drop if distance is crazy short

trips_before = len(combined_trips)
combined_trips = combined_trips[combined_trips['distance_miles']>0.1].copy()

print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 161874 After: 161711


In [15]:
# drop if distance is crazy long

trips_before = len(combined_trips)
combined_trips = combined_trips[combined_trips['distance_miles']<50].copy()

print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 161711 After: 161709


In [16]:
# drop if time is crazy short

trips_before = len(combined_trips)
combined_trips = combined_trips[combined_trips['duration_minutes']>1].copy()

print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 161709 After: 161516


In [17]:
# drop if time is crazy long

trips_before = len(combined_trips)
combined_trips = combined_trips[combined_trips['duration_minutes']<180].copy()

print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 161516 After: 161454


In [18]:
# drop if observed fare is crazy high

trips_before = len(combined_trips)
combined_trips = combined_trips[(combined_trips['obs_fare']<60) | (combined_trips['obs_fare'].isna())].copy()

print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 161454 After: 161252


In [19]:
# drop if observed additional charges are crazy high

trips_before = len(combined_trips)
combined_trips = combined_trips[(combined_trips['obs_additional_charges']<100) | (combined_trips['obs_fare'].isna())].copy()

print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 161252 After: 161249


In [20]:
# set availability rules for transit

# transit is not available if the router can't find a path
combined_trips['transit_avail'] = np.where(combined_trips['transit_time']<999, 1, 0)

# drop trips that choose an unavailable alternative
trips_before = len(combined_trips)
combined_trips = combined_trips[(combined_trips['transit_avail']==1) | (combined_trips['mode']!='transit')].copy()
print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 161249 After: 161249


In [21]:
# set availability rules for walk

# walk is not avaiable if it would take more than 1 hour to walk
combined_trips['walk_avail'] = np.where(combined_trips['walk_time']<60, 1, 0)

# drop trips that choose an unavailable alternative
trips_before = len(combined_trips)
combined_trips = combined_trips[(combined_trips['walk_avail']==1) | (combined_trips['mode']!='walk')].copy()
print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 161249 After: 161249


In [22]:
# set availability rules for transit or walk

combined_trips['transit_or_walk_avail'] = combined_trips[['transit_avail', 'walk_avail']].max(axis=1)

# drop trips that choose an unavailable alternative
trips_before = len(combined_trips)
combined_trips = combined_trips[(combined_trips['transit_or_walk_avail']==1) | (combined_trips['mode']=='tnc')].copy()
print("Before: " + str(trips_before) + " After: " + str(len(combined_trips)))

Before: 161249 After: 161249


# Use observed TNC time and cost where available


In [23]:
# use the observed TNC time and fare where they are avialable

combined_trips['tnc_time_2'] = np.where((combined_trips['mode']=='tnc') & (combined_trips['duration_minutes'].notna()), 5+combined_trips['duration_minutes'], 5+combined_trips['car_ivt'])
combined_trips['tnc_fare_2'] = np.where((combined_trips['mode']=='tnc') & (combined_trips['obs_fare'].notna()), combined_trips['obs_fare'], combined_trips['tnc_fare'])


In [24]:
# map time period to a numerical value for inclusion in biogeme
mapping = {'night' : 1, 'am_peak' : 2, 'midday' : 3, 'pm_peak' : 4, 'evening' : 5}
combined_trips['time_period_num'] = combined_trips['time_period'].map(mapping)

# keep only the TNC trips and aggregate by o_tract, d_tract, and time_period_num
tnc_trips = combined_trips[combined_trips['mode']=='tnc'].copy()

# aggregate by o_tract, d_tract, and time_period_num
los_lookup = tnc_trips.groupby(by=['o_tract_2020', 'd_tract_2020', 'time_period_num']).agg({'duration_minutes' : 'mean', 'obs_fare': 'mean'})
los_lookup = los_lookup.rename(columns={'duration_minutes' : 'avg_obs_tnc_time', 'obs_fare' : 'avg_obs_tnc_fare'})
los_lookup.head()

avg_obs_tnc_time  avg_obs_tnc_fare
o_tract_2020 d_tract_2020 time_period_num                                    
17031010100  17031010100  2                       14.266667              8.75
                          3                        3.983333              7.50
                          4                        5.383333              7.50
             17031010201  1                        3.450000              5.00
             17031010202  2                        2.900000              5.00

In [25]:
# merge back to the main estimation file
combined_trips = combined_trips.merge(los_lookup, on=['o_tract_2020', 'd_tract_2020', 'time_period_num'], how='left')
combined_trips.head()

,hh_id,person_id,person_num,day_id,day_num,depart_date,o_tract_2020,d_tract_2020,linked_trip_id,linked_trip_num,linked_trip_mode,linked_trip_weight,linked_trip_mode_labeled,mode,mode2,distance_miles,duration_minutes,o_district,d_district,o_community,d_community,time_period,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,walk_faster_than_transit,transit_or_walk_fare,income_broad,income_labeled,hh_share_inc_under_100k,hh_share_inc_over_100k,hh_share_inc_under_100k_dropoff,hh_share_inc_over_100k_dropoff,tnc_trip_id,obs_fare,obs_tip,obs_additional_charges,transit_avail,walk_avail,transit_or_walk_avail,tnc_time_2,tnc_fare_2,time_period_num,avg_obs_tnc_time,avg_obs_tnc_fare
0,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031081500,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.810270,20.0,Downtown,Downtown,32.0,8.0,midday,3.738333,6.186942,5,11.186942,6.707731,2.5,16.205401,22.0,16.205401,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0.321678,0.678322,NaN,NaN,NaN,NaN,1,1,1,11.186942,6.707731,3,NaN,NaN
1,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081500,17031081403,2.400012e+15,2.0,15.0,1853.792592,Walk,walk,walk,0.338027,28.0,Downtown,Downtown,8.0,8.0,midday,1.660000,2.747300,5,7.747300,4.798943,2.5,6.760535,7.0,6.760535,True,0.0,5.0,"$150,000 or more",0.321678,0.678322,0.460539,0.539461,NaN,NaN,NaN,NaN,1,1,1,7.747300,4.798943,3,7.157143,9.642857
2,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081403,17031320101,2.400012e+15,3.0,15.0,1853.792592,Walk,walk,walk,0.549293,15.0,Downtown,Downtown,8.0,32.0,midday,3.421667,5.662858,5,10.662858,6.244886,2.5,10.985870,23.0,10.985870,True,0.0,5.0,"$150,000 or more",0.460539,0.539461,0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,10.662858,6.244886,3,NaN,NaN
3,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320101,2.400012e+15,4.0,15.0,1853.792592,Walk,walk,walk,0.319386,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.167457,2.5,6.387712,12.0,6.387712,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,8.643758,5.167457,3,NaN,NaN
4,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320102,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.751861,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.561010,2.5,15.037220,12.0,12.000000,False,2.5,5.0,"$150,000 or more",0.341000,0.659000,0.519520,0.480480,NaN,NaN,NaN,NaN,1,1,1,8.643758,5.561010,3,NaN,NaN


In [26]:
# summarize the percent of transit and walk records for which we have average observed TNC times and fares

combined_trips['tnc_observed'] = combined_trips['avg_obs_tnc_time'].notna()
display(pd.crosstab(combined_trips['mode'], combined_trips['tnc_observed'], margins=True).style.format('{:,.0f}'))


tnc_observed,False,True,All
mode,,,
tnc,0,"156,150","156,150"
transit,"1,017",566,"1,583"
walk,"1,852","1,664","3,516"
All,"2,869","158,380","161,249"


In [27]:
# update with observed values where available
combined_trips['tnc_time_3'] = np.where((combined_trips['mode']!='tnc') & (combined_trips['tnc_observed']), 5 + combined_trips['avg_obs_tnc_time'], combined_trips['tnc_time_2'])
combined_trips['tnc_fare_3'] = np.where((combined_trips['mode']!='tnc') & (combined_trips['tnc_observed']), combined_trips['avg_obs_tnc_fare'], combined_trips['tnc_fare_2'])

combined_trips.head(20)

,hh_id,person_id,person_num,day_id,day_num,depart_date,o_tract_2020,d_tract_2020,linked_trip_id,linked_trip_num,linked_trip_mode,linked_trip_weight,linked_trip_mode_labeled,mode,mode2,distance_miles,duration_minutes,o_district,d_district,o_community,d_community,time_period,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,walk_faster_than_transit,transit_or_walk_fare,income_broad,income_labeled,hh_share_inc_under_100k,hh_share_inc_over_100k,hh_share_inc_under_100k_dropoff,hh_share_inc_over_100k_dropoff,tnc_trip_id,obs_fare,obs_tip,obs_additional_charges,transit_avail,walk_avail,transit_or_walk_avail,tnc_time_2,tnc_fare_2,time_period_num,avg_obs_tnc_time,avg_obs_tnc_fare,tnc_observed,tnc_time_3,tnc_fare_3
0,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031081500,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.810270,20.0,Downtown,Downtown,32.0,8.0,midday,3.738333,6.186942,5,11.186942,6.707731,2.5,16.205401,22.0,16.205401,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0.321678,0.678322,NaN,NaN,NaN,NaN,1,1,1,11.186942,6.707731,3,NaN,NaN,False,11.186942,6.707731
1,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081500,17031081403,2.400012e+15,2.0,15.0,1853.792592,Walk,walk,walk,0.338027,28.0,Downtown,Downtown,8.0,8.0,midday,1.660000,2.747300,5,7.747300,4.798943,2.5,6.760535,7.0,6.760535,True,0.0,5.0,"$150,000 or more",0.321678,0.678322,0.460539,0.539461,NaN,NaN,NaN,NaN,1,1,1,7.747300,4.798943,3,7.157143,9.642857,True,12.157143,9.642857
2,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081403,17031320101,2.400012e+15,3.0,15.0,1853.792592,Walk,walk,walk,0.549293,15.0,Downtown,Downtown,8.0,32.0,midday,3.421667,5.662858,5,10.662858,6.244886,2.5,10.985870,23.0,10.985870,True,0.0,5.0,"$150,000 or more",0.460539,0.539461,0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,10.662858,6.244886,3,NaN,NaN,False,10.662858,6.244886
3,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320101,2.400012e+15,4.0,15.0,1853.792592,Walk,walk,walk,0.319386,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.167457,2.5,6.387712,12.0,6.387712,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,8.643758,5.167457,3,NaN,NaN,False,8.643758,5.167457
4,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320102,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.751861,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.561010,2.5,15.037220,12.0,12.000000,False,2.5,5.0,"$150,000 or more",0.341000,0.659000,0.519520,0.480480,NaN,NaN,NaN,NaN,1,1,1,8.643758,5.561010,3,NaN,NaN,False,8.643758,5.561010
5,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031320102,17031320101,2.400012e+15,2.0,15.0,1853.792592,Walk,walk,walk,0.751861,17.0,Downtown,Downtown,32.0,32.0,midday,2.358333,3.903042,5,8.903042,5.672501,2.5,15.037220,12.0,12.000000,False,2.5,5.0,"$150,000 or more",0.519520,0.480480,0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,8.903042,5.672501,3,NaN,NaN,False,8.903042,5.672501
6,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031320101,17031081800,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,1.728659,40.0,Downtown,Downtown,32.0,8.0,midday,4.898333,8.106742,5,13.106742,8.368979,2.5,34.573179,33.0,33.000000,False,2.5,5.0,"$150,000 or more",0.341000,0.659000,0.262262,0.737738,NaN,NaN,NaN,NaN,1,1,1,13.106742,8.368979,3,NaN,NaN,False,13.106742,8.368979
7,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031081800,17031320101,2.400012e+15,2.0,15.0,1853.792592,Walk,walk,walk,1.728659,45.0,Downtown,Downtown,8.0,32.0,midday,5.696667,9.427983,5,14.427983,8.937112,2.5,34.573179,37.0,34.573179,True,0.0,5.0,"$150,000 or more",0.262262,0.737738,0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,14.427983,8.937112,3,NaN,NaN,False,14.427983,8.937112
8,24000638.0,2.400064e+09,1.0,2.400064e+11,3.0,2024-05-21,17031150300,17031150200,2.400064e+15,3.0

# Summarize the data

In [28]:
# update availability

# transit must be less than 2 hours
combined_trips['transit_avail'] = np.where(combined_trips['transit_time']<120, 1, 0)

# walk must be less than 30 minutes
combined_trips['walk_avail'] = np.where(combined_trips['walk_time']<30, 1, 0)

In [29]:
# calculated fields for binary choice

# note whether transit or walk is faster
combined_trips['walk_faster_than_transit'] = (combined_trips['walk_time'] < combined_trips['transit_time']) & (combined_trips['walk_avail'])

# determine the faster of the walk or transit time for each trip
combined_trips['transit_or_walk_time'] = np.where(combined_trips['walk_faster_than_transit'], combined_trips['walk_time'], combined_trips['transit_time'])

# if walk is faster, set the transit_or_walk_fare to zero
combined_trips['transit_or_walk_fare'] = np.where(combined_trips['walk_faster_than_transit'], 0, combined_trips['transit_fare'])


In [30]:
combined_trips.head()

,hh_id,person_id,person_num,day_id,day_num,depart_date,o_tract_2020,d_tract_2020,linked_trip_id,linked_trip_num,linked_trip_mode,linked_trip_weight,linked_trip_mode_labeled,mode,mode2,distance_miles,duration_minutes,o_district,d_district,o_community,d_community,time_period,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,walk_faster_than_transit,transit_or_walk_fare,income_broad,income_labeled,hh_share_inc_under_100k,hh_share_inc_over_100k,hh_share_inc_under_100k_dropoff,hh_share_inc_over_100k_dropoff,tnc_trip_id,obs_fare,obs_tip,obs_additional_charges,transit_avail,walk_avail,transit_or_walk_avail,tnc_time_2,tnc_fare_2,time_period_num,avg_obs_tnc_time,avg_obs_tnc_fare,tnc_observed,tnc_time_3,tnc_fare_3
0,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031081500,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.810270,20.0,Downtown,Downtown,32.0,8.0,midday,3.738333,6.186942,5,11.186942,6.707731,2.5,16.205401,22.0,16.205401,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0.321678,0.678322,NaN,NaN,NaN,NaN,1,1,1,11.186942,6.707731,3,NaN,NaN,False,11.186942,6.707731
1,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081500,17031081403,2.400012e+15,2.0,15.0,1853.792592,Walk,walk,walk,0.338027,28.0,Downtown,Downtown,8.0,8.0,midday,1.660000,2.747300,5,7.747300,4.798943,2.5,6.760535,7.0,6.760535,True,0.0,5.0,"$150,000 or more",0.321678,0.678322,0.460539,0.539461,NaN,NaN,NaN,NaN,1,1,1,7.747300,4.798943,3,7.157143,9.642857,True,12.157143,9.642857
2,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081403,17031320101,2.400012e+15,3.0,15.0,1853.792592,Walk,walk,walk,0.549293,15.0,Downtown,Downtown,8.0,32.0,midday,3.421667,5.662858,5,10.662858,6.244886,2.5,10.985870,23.0,10.985870,True,0.0,5.0,"$150,000 or more",0.460539,0.539461,0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,10.662858,6.244886,3,NaN,NaN,False,10.662858,6.244886
3,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320101,2.400012e+15,4.0,15.0,1853.792592,Walk,walk,walk,0.319386,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.167457,2.5,6.387712,12.0,6.387712,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,8.643758,5.167457,3,NaN,NaN,False,8.643758,5.167457
4,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320102,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.751861,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.561010,2.5,15.037220,12.0,12.000000,False,2.5,5.0,"$150,000 or more",0.341000,0.659000,0.519520,0.480480,NaN,NaN,NaN,NaN,1,1,1,8.643758,5.561010,3,NaN,NaN,False,8.643758,5.561010


In [31]:
# calculate the time and cost differences for TNC vs transit/walk
combined_trips['tnc_time_minus_transit_walk'] = combined_trips['tnc_time_3'] - combined_trips['transit_or_walk_time']
combined_trips['tnc_cost_minus_transit_walk'] = combined_trips['tnc_fare_3'] - combined_trips['transit_or_walk_fare']

In [32]:
combined_trips.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

,hh_id,person_id,person_num,day_id,day_num,o_tract_2020,d_tract_2020,linked_trip_id,linked_trip_num,linked_trip_mode,linked_trip_weight,distance_miles,duration_minutes,o_community,d_community,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,transit_or_walk_fare,income_broad,hh_share_inc_under_100k,hh_share_inc_over_100k,hh_share_inc_under_100k_dropoff,hh_share_inc_over_100k_dropoff,obs_fare,obs_tip,obs_additional_charges,transit_avail,walk_avail,transit_or_walk_avail,tnc_time_2,tnc_fare_2,time_period_num,avg_obs_tnc_time,avg_obs_tnc_fare,tnc_time_3,tnc_fare_3,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk
count,5.099000e+03,5.099000e+03,5099.000000,5.099000e+03,5099.000000,1.612490e+05,1.612490e+05,5.099000e+03,5099.000000,5099.000000,161249.000000,161249.000000,161249.000000,161249.000000,161249.000000,161249.000000,161249.000000,161249.0,161249.000000,161249.000000,161249.0,161249.000000,161249.000000,161249.000000,161249.000000,5099.000000,161105.000000,161105.000000,161204.000000,161204.000000,155943.000000,155943.000000,155943.000000,161249.000000,161249.000000,161249.000000,161249.000000,161249.000000,161249.000000,158380.000000,158268.000000,161249.000000,161249.000000,161249.000000,161249.000000
mean,2.407447e+07,2.407447e+09,1.396352,2.407447e+11,2.907041,1.703139e+10,1.703140e+10,2.407447e+15,1.943518,12.300255,17.122182,4.339538,15.996306,27.283078,27.599855,8.109870,13.618073,5.0,18.618073,13.954632,2.5,86.790756,78.316554,77.229451,2.161827,45.116297,0.537067,0.462933,0.534419,0.465581,13.238507,0.778304,3.799590,0.936818,0.220256,0.965147,20.509310,13.103213,3.109892,15.657396,13.190330,20.549854,13.134835,-56.679598,10.973008
std,6.277273e+04,6.277273e+06,0.822398,6.277273e+08,2.178299,3.097480e+05,3.121104e+05,6.277273e+12,1.328958,4.057823,229.449779,3.839200,11.098986,20.088293,20.118105,5.385814,9.218202,0.0,9.218202,7.569872,0.0,76.784006,176.969125,177.294252,0.855029,200.161399,0.204144,0.204144,0.201932,0.201932,7.513593,1.847676,2.824076,0.243290,0.414420,0.183408,10.427333,7.521972,1.269838,10.141244,6.956109,10.419341,7.513074,175.476979,7.296812
min,2.400012e+07,2.400012e+09,1.000000,2.400012e+11,1.000000,1.703101e+10,1.703101e+10,2.400012e+15,1.000000,3.000000,0.000000,0.100041,1.016667,1.000000,1.000000,0.000000,0.000000,5.0,5.000000,3.709778,2.5,2.000820,4.000000,2.000820,0.000000,1.000000,0.134000,0.000000,0.134000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.623383,0.000000,1.000000,1.016667,0.000000,5.623383,0.000000,-984.500000,-2.500000
1%,2.400216e+07,2.400216e+09,1.000000,2.400216e+11,1.000000,1.703101e+10,1.703101e+10,2.400216e+15,1.000000,3.000000,0.000000,0.338648,2.666667,1.000000,1.000000,0.000000,0.000000,5.0,5.000000,4.788898,2.5,6.772963,9.000000,6.000000,0.000000,1.000000,0.202000,0.051000,0.202000,0.051000,2.500000,0.000000,1.170000,0.000000,0.000000,0.000000,7.466667,2.500000,1.000000,2.991667,2.500000,7.550000,2.500000,-975.466667,0.000000
5%,2.401379e+07,2.401379e+09,1.000000,2.401379e+11,1.000000,1.703103e+10,1.703103e+10,2.401379e+15,1.000000,6.000000,1.000000,0.700000,4.250000,4.000000,4.000000,2.045000,3.318275,5.0,8.318275,5.726595,2.5,14.000000,15.000000,12.000000,0.000000,1.000000,0.260521,0.115115,0.262262,0.115000,5.000000,0.000000,1.230000,0.000000,0.000000,1.000000,8.966667,5.000000,1.000000,4.641607,5.000000,9.066667,5.000000,-122.583333,2.500000
50%,2.407244e+07,2.407244e+09,1.000000,2.407244e+11,2.000000,1.703128e+10,1.703129e+10,2.407244e+15,2.000000,15.000000,1.000000,3.100000,13.233333,26.000000,28.000000,6.771667,11.322502,5.0,16.322502,11.921475,2.5,62.000000,40.000000,39.000000,2.500000,3.000000,0.521000,0.479000,0.519520,0.480480,12.500000,0.000000,2.960000,1.000000,0.000000,1.000000,17.983333,12.500000,3.000000,13.050000,11.785714,18.000000,12.500000,-18.666667,10.000000
95%,2.414789e+07,2.414789e+09,3.000000,2.414789e+11,7.000000,1.703184e+10,1.703184e+10,2.414789e

In [33]:
print('All Trips')
display(combined_trips[['distance_miles', 
                     'tnc_time', 'tnc_time_3', 'walk_time', 'transit_time', 
                     'tnc_fare', 'tnc_fare_3', 'transit_fare', 
                     'transit_or_walk_time', 'transit_or_walk_fare', 
                     'tnc_time_minus_transit_walk', 'tnc_cost_minus_transit_walk', 
                     'transit_avail', 'walk_avail'
                     ]].describe(percentiles=[0.05, 0.5, 0.95]).style.format('{:.1f}'))

print('\nTNC Trips')
tnc = combined_trips[combined_trips['mode']=='tnc']
display(tnc[['distance_miles', 
             'tnc_time', 'tnc_time_3', 'walk_time', 'transit_time', 
             'tnc_fare', 'tnc_fare_3', 'transit_fare', 
             'transit_or_walk_time', 'transit_or_walk_fare', 
             'transit_or_walk_time', 'transit_or_walk_fare', 
             'tnc_time_minus_transit_walk', 'tnc_cost_minus_transit_walk'
             ]].describe(percentiles=[0.05, 0.5, 0.95]).style.format('{:.1f}'))

print('\nTransit Trips')
transit = combined_trips[combined_trips['mode']=='transit']
display(transit[['distance_miles', 
                 'tnc_time', 'tnc_time_3', 'walk_time', 'transit_time', 
                 'tnc_fare', 'tnc_fare_3', 'transit_fare', 
                 'transit_or_walk_time', 'transit_or_walk_fare', 
                 'tnc_time_minus_transit_walk', 'tnc_cost_minus_transit_walk'
                 ]].describe(percentiles=[0.05, 0.5, 0.95]).style.format('{:.1f}'))

print('\nWalk Trips')
walk = combined_trips[combined_trips['mode']=='walk']
display(walk[['distance_miles', 
              'tnc_time', 'tnc_time_3', 'walk_time', 'transit_time', 
              'tnc_fare', 'tnc_fare_3', 'transit_fare', 
              'transit_or_walk_time', 'transit_or_walk_fare', 
              'tnc_time_minus_transit_walk', 'tnc_cost_minus_transit_walk'
              ]].describe(percentiles=[0.05, 0.5, 0.95]).style.format('{:.1f}'))


All Trips


,distance_miles,tnc_time,tnc_time_3,walk_time,transit_time,tnc_fare,tnc_fare_3,transit_fare,transit_or_walk_time,transit_or_walk_fare,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk,transit_avail,walk_avail
count,161249.0,161249.0,161249.0,161249.0,161249.0,161249.0,161249.0,161249.0,161249.0,161249.0,161249.0,161249.0,161249.0,161249.0
mean,4.3,18.6,20.5,86.8,78.3,14.0,13.1,2.5,77.2,2.2,-56.7,11.0,0.9,0.2
std,3.8,9.2,10.4,76.8,177.0,7.6,7.5,0.0,177.3,0.9,175.5,7.3,0.2,0.4
min,0.1,5.0,5.6,2.0,4.0,3.7,0.0,2.5,2.0,0.0,-984.5,-2.5,0.0,0.0
5%,0.7,8.3,9.1,14.0,15.0,5.7,5.0,2.5,12.0,0.0,-122.6,2.5,0.0,0.0
50%,3.1,16.3,18.0,62.0,40.0,11.9,12.5,2.5,39.0,2.5,-18.7,10.0,1.0,0.0
95%,12.2,36.5,40.8,244.0,143.0,28.8,27.5,2.5,143.0,2.5,-1.0,25.0,1.0,1.0
max,39.6,92.7,158.6,792.0,999.0,83.0,57.5,2.5,999.0,2.5,122.5,55.0,1.0,1.0



TNC Trips


,distance_miles,tnc_time,tnc_time_3,walk_time,transit_time,tnc_fare,tnc_fare_3,transit_fare,transit_or_walk_time,transit_or_walk_fare,transit_or_walk_time,transit_or_walk_fare,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk
count,156150.0,156150.0,156150.0,156150.0,156150.0,156150.0,156150.0,156150.0,156150.0,156150.0,156150.0,156150.0,156150.0,156150.0
mean,4.4,18.8,20.7,88.4,80.1,14.1,13.2,2.5,79.1,2.2,79.1,2.2,-58.3,11.1
std,3.8,9.2,10.4,76.9,179.5,7.6,7.5,0.0,179.8,0.8,179.8,0.8,178.1,7.3
min,0.2,5.0,6.0,4.0,4.0,3.9,0.0,2.5,4.0,0.0,4.0,0.0,-984.5,-2.5
5%,0.8,8.5,9.3,16.0,16.0,5.9,5.0,2.5,14.0,0.0,14.0,0.0,-125.5,2.5
50%,3.2,16.5,18.2,64.0,40.0,12.1,12.5,2.5,40.0,2.5,40.0,2.5,-19.1,10.0
95%,12.3,36.7,41.0,246.0,146.0,29.0,27.5,2.5,146.0,2.5,146.0,2.5,-2.1,25.0
max,39.6,92.7,158.6,792.0,999.0,83.0,57.5,2.5,999.0,2.5,999.0,2.5,122.5,55.0



Transit Trips


,distance_miles,tnc_time,tnc_time_3,walk_time,transit_time,tnc_fare,tnc_fare_3,transit_fare,transit_or_walk_time,transit_or_walk_fare,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk
count,1583.0,1583.0,1583.0,1583.0,1583.0,1583.0,1583.0,1583.0,1583.0,1583.0,1583.0,1583.0
mean,4.9,22.2,23.3,98.8,43.3,15.2,15.8,2.5,42.7,2.3,-19.4,13.5
std,3.3,9.5,9.9,66.5,17.7,6.9,7.1,0.0,18.4,0.7,14.5,6.9
min,0.1,6.6,6.6,2.7,5.0,4.3,2.5,2.5,2.7,0.0,-154.6,0.0
5%,0.9,9.7,10.7,18.1,20.0,6.3,6.7,2.5,16.3,0.0,-36.8,5.0
50%,4.4,21.1,21.9,88.3,42.0,14.4,15.0,2.5,42.0,2.5,-18.2,12.5
95%,10.7,39.1,41.7,214.7,72.0,27.9,29.7,2.5,72.0,2.5,-2.2,27.2
max,33.7,82.1,82.1,673.2,175.0,50.2,50.2,2.5,175.0,2.5,19.7,47.7



Walk Trips


,distance_miles,tnc_time,tnc_time_3,walk_time,transit_time,tnc_fare,tnc_fare_3,transit_fare,transit_or_walk_time,transit_or_walk_fare,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk
count,3516.0,3516.0,3516.0,3516.0,3516.0,3516.0,3516.0,3516.0,3516.0,3516.0,3516.0,3516.0
mean,0.6,9.3,10.7,11.7,17.0,5.7,6.9,2.5,10.6,0.5,0.0,6.3
std,0.4,2.2,4.4,8.9,7.8,1.2,2.9,0.0,7.3,1.0,7.7,3.0
min,0.1,5.6,5.6,2.0,4.0,3.7,2.5,2.5,2.0,0.0,-50.0,0.0
5%,0.2,6.8,7.0,3.3,6.0,4.4,4.4,2.5,3.3,0.0,-12.4,2.5
50%,0.4,8.8,10.0,9.0,16.0,5.4,6.1,2.5,8.6,0.0,0.9,5.6
95%,1.5,13.4,16.0,29.7,31.0,8.0,11.6,2.5,24.0,2.5,8.5,11.2
max,3.0,27.0,91.2,59.9,64.0,14.5,45.0,2.5,64.0,2.5,84.2,42.5


In [34]:
# print both the absolute totals and the mode shares

# TNC
tnc = combined_trips[combined_trips['mode']=='tnc']
print('TNC Trips - Observations')
display(pd.crosstab(tnc['o_district'], tnc['d_district'], margins=True).style.format('{:,.0f}'))

print('TNC Trips - Weighted & Expanded')
tnc_crosstab = pd.crosstab(tnc['o_district'], tnc['d_district'], values=tnc['linked_trip_weight'], aggfunc='sum', margins=True)
display(tnc_crosstab.style.format('{:,.0f}'))

# Transit
transit = combined_trips[combined_trips['mode']=='transit']
print('Transit Trips - Observations')
display(pd.crosstab(transit['o_district'], transit['d_district'], margins=True).style.format('{:,.0f}'))

print('Transit Trips - Weighted & Expanded')
transit_crosstab = pd.crosstab(transit['o_district'], transit['d_district'], values=transit['linked_trip_weight'], aggfunc='sum', margins=True)
display(transit_crosstab.style.format('{:,.0f}'))

# Walk
walk = combined_trips[(combined_trips['mode']=='walk')]
print('Walk Trips - Observations')
display(pd.crosstab(walk['o_district'], walk['d_district'], margins=True).style.format('{:,.0f}'))

print('Walk Trips - Weighted & Expanded')
walk_crosstab = pd.crosstab(walk['o_district'], walk['d_district'], values=walk['linked_trip_weight'], aggfunc='sum', margins=True)
display(walk_crosstab.style.format('{:,.0f}'))


# now print the mode shares with the updated walk, excluding both <0.5 miles and loops
total_crosstab = tnc_crosstab + transit_crosstab + walk_crosstab
tnc_share = tnc_crosstab / total_crosstab
transit_share = transit_crosstab / total_crosstab
walk_share = walk_crosstab / total_crosstab

print('TNC Mode Share')
display(tnc_share.style.format('{:.1%}'))

print('Transit Mode Share')
display(transit_share.style.format('{:.1%}'))

print('Walk Mode Share')
display(walk_share.style.format('{:.1%}'))

TNC Trips - Observations


d_district,Downtown,Other,Tourist,All
o_district,,,,
Downtown,"26,926","19,143",428,"46,497"
Other,"21,027","86,703",506,"108,236"
Tourist,514,899,4,"1,417"
All,"48,467","106,745",938,"156,150"


TNC Trips - Weighted & Expanded


d_district,Downtown,Other,Tourist,All
o_district,,,,
Downtown,"26,926","19,143",428,"46,497"
Other,"21,027","86,703",506,"108,236"
Tourist,514,899,4,"1,417"
All,"48,467","106,745",938,"156,150"


Transit Trips - Observations


...

d_district,Downtown,Other,Tourist,All
o_district,,,,
Downtown,5.7%,7.6%,7.8%,6.4%
Other,7.1%,5.1%,nan%,5.4%
Tourist,8.8%,7.5%,nan%,7.3%
All,6.3%,5.4%,11.0%,5.7%


Transit Mode Share


d_district,Downtown,Other,Tourist,All
o_district,,,,
Downtown,21.6%,73.1%,22.8%,39.5%
Other,80.3%,29.1%,nan%,36.7%
Tourist,20.3%,80.2%,nan%,55.7%
All,44.2%,35.1%,25.6%,37.6%


Walk Mode Share


d_district,Downtown,Other,Tourist,All
o_district,,,,
Downtown,72.7%,19.3%,69.5%,54.1%
Other,12.6%,65.8%,nan%,57.9%
Tourist,70.9%,12.4%,nan%,37.0%
All,49.6%,59.5%,63.4%,56.8%


I'm not sure why the graphs look like that, but I can dig in later.  

# write the output

In [35]:
combined_trips.to_csv('out/combined_estimation_file.csv')